In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
import tqdm
from jppype import vscode_theme
from torch import nn
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
# torch.backends.fp32_precision = "tf32"

In [5]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

In [6]:
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024)
dataset.augment = True

Found 215 branch digraphs...


Processing...
Processing dataset:  24%|██▎       | 51/215 [00:14<00:35,  4.59it/s]/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vgraph.py:529: UserWarning: The geometric data contains duplicated nodes coordinates.
  self.check_integrity("warn" if check_integrity else "skip")
Processing dataset:  40%|███▉      | 85/215 [00:23<00:43,  2.98it/s]/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vgraph.py:529: UserWarning: The geometric data contains duplicated nodes coordinates.
  self.check_integrity("warn" if check_integrity else "skip")
Processing dataset:  68%|██████▊   | 146/215 [00:37<00:15,  4.38it/s]/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vgraph.py:529: UserWarning: The geometric data contains duplicated nodes coordinates.
  self.check_integrity("warn" if check_integrity else "skip")
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_too

In [7]:
m, digraph, _ = dataset.draw_jppype(0, augment=True, test=True)
m

[ WARN:0@0.335] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [8]:
from fundus_vessels_toolkit.segment_to_graph.models.digraph_model import (
    BranchDigraphModel,
    BranchFeaturesEfficientNetV2S,
    Gatv2GCN,
)
import torch.nn.functional as F

torch._dynamo.config.capture_dynamic_output_shape_ops = True
model = BranchDigraphModel(BranchFeaturesEfficientNetV2S(), Gatv2GCN(n_in=784, n_out=512))
model = model.cuda()
# model = torch.compile(model, dynamic=True)


In [9]:
av_nll_loss = nn.NLLLoss()
dir_bce_loss = nn.BCEWithLogitsLoss()
line_ce_loss = nn.CrossEntropyLoss()

for i, batch in enumerate(tqdm.tqdm(DataLoader(dataset, batch_size=3, num_workers=5))):
    av_p, dir_p, edge_p, valid_lines, valid_roots = model(batch.cuda())
    # AV loss
    target_av_p = torch.cat([1 - batch.branch_av_p.sum(dim=-1, keepdim=True), batch.branch_av_p], dim=-1)
    target_av = torch.argmax(target_av_p, dim=-1)
    av_loss = av_nll_loss(F.log_softmax(av_p, dim=1), target_av)

    # Dir and line losses
    dir_loss = dir_bce_loss(dir_p, batch.branch_dir)
    edge_p_gt = torch.cat([batch.edge_p[valid_lines], batch.branch_root_p[valid_roots]], dim=0)
    line_loss = line_ce_loss(edge_p, edge_p_gt)

    loss = av_loss + dir_loss + line_loss
    loss.backward()

100%|██████████| 72/72 [00:19<00:00,  3.68it/s]
